# Validation Ranking Walkthrough

This notebook inspects the first end-to-end validation run. It compares popularity, LightGCN average aggregation, and conflict-aware aggregation under the same candidate rules. This is an integration check, not the final experiment.

In [ ]:
import json
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
output_dir = project_root / "outputs/lightgcn_subset_validation"
report_path = output_dir / "validation_report.json"
if not report_path.exists():
    raise FileNotFoundError(
        "Run scripts/train_lightgcn_subset.py and scripts/evaluate_lightgcn_subset.py first."
    )

## 1. Check the evaluation scope

The target-coverage value measures how many validation positives remain scoreable after restricting evaluation to the subset catalogue and removing movies seen by either pair member. Low coverage means the subset experiment cannot answer the final research question reliably.

In [ ]:
report = json.loads(report_path.read_text(encoding="utf-8"))
scope = {
    "purpose": report["purpose"],
    "pairs": report["pairs"],
    "candidate_movies": report["candidate_movies"],
    "pair_validation_target_coverage": report["pair_validation_target_coverage"],
}
scope

## 2. Compare methods

Minimum-member NDCG is the primary fairness-oriented relevance metric. Average NDCG measures overall pair utility, the NDCG gap measures within-pair inequality, and catalogue coverage is the beyond-accuracy metric.

In [ ]:
method_table = pd.DataFrame(report["methods"])
columns = [
    "method",
    "meanMinimumNDCG@10",
    "meanAverageNDCG@10",
    "meanNDCGGap@10",
    "meanAverageRecall@10",
    "catalogueCoverage@10",
]
method_table[columns].sort_values(
    ["meanMinimumNDCG@10", "meanAverageNDCG@10"],
    ascending=False,
).reset_index(drop=True)

## 3. Interpret model selection cautiously

A tie on the primary metric means the configured conflict penalty has not demonstrated an improvement. The reported selected method is only the deterministic tie-break result.

In [ ]:
print("Primary metric:", report["selection_metric"])
print("Primary value:", report["selection_metric_value"])
print("Methods tied on the primary metric:")
for method in report["methods_tied_on_selection_metric"]:
    print(" -", method)
print("Tie-break selection:", report["selected_personalized_method"])

## 4. Inspect pair-level behavior

Aggregate means can hide whether only a few pairs receive useful recommendations. Pair-level metrics expose which groups get hits for both members and which groups get none.

In [ ]:
pair_metrics = pd.read_csv(output_dir / "pair_metrics.csv.gz")
pair_metrics.sort_values(
    ["minimumNDCG@10", "averageNDCG@10"],
    ascending=False,
).head(12)

## 5. Inspect one pair's ranked lists

The same pair and candidate policy are used for every method, so differences below come from scoring and aggregation rather than different candidate access.

In [ ]:
recommendations = pd.read_csv(output_dir / "recommendations.csv.gz")
example_pair = int(recommendations["pairId"].iloc[0])
recommendations.loc[
    recommendations["pairId"] == example_pair,
    ["method", "rank", "movieId", "groupScore", "scoreA", "scoreB"],
].sort_values(["method", "rank"])

## 6. Next experimental step

Increase the number of evaluation pairs and graph context, improve validation-target coverage, and train long enough for stable personalized scores. Keep K=10 as the research metric. A larger diagnostic K may be inspected separately to understand sparsity, but it must not replace the predeclared primary metric after seeing results.